<a href="https://colab.research.google.com/github/Rayoyo/NLP-Translator-JA-EN/blob/main/nlp_translator_main_VER3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Enviroment setup

In [33]:
# Verifica GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA disponibile: True
GPU: Tesla T4
VRAM: 15.6 GB


In [34]:
# To keep Colab "alive" avoiding inactivity timeout
# Esegui in una cella separata e lasciala girare
import time
from IPython.display import display, Javascript

def keep_alive():
    display(Javascript('''
        function keepAlive() {
            setInterval(() => {
                document.querySelector("colab-toolbar-button#connect").click();
                console.log("Keep alive");
            }, 60000);
        }
        keepAlive();
    '''))

# keep_alive()  # Decommenta se vuoi provare

In [35]:
# Install dependencies
!pip install -q sentencepiece sacrebleu transformers gradio datasets tqdm

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import torch
import torch.nn as nn
import sys
import os

---
## 2. Clone repository from Github

In [ ]:
# Clona la repo (oppure carica i file manualmente)
# !git clone https://github.com/TUO_USERNAME/PROGETTO-TRADUTTORE.git
# sys.path.append('/content/PROGETTO-TRADUTTORE')

# OPPURE carica src/ manualmente su Colab
# Assumiamo che i file siano in /content/drive/MyDrive/progetto/

In [38]:
!git clone https://github.com/Rayoyo/NLP-Translator-JA-EN.git
%cd NLP-Translator-JA-EN

import sys
sys.path.append('/content/NLP-Translator-JA-EN')

print("Repo clonata!")
!ls -la src/

Cloning into 'NLP-Translator-JA-EN'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 131 (delta 43), reused 83 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 1.25 MiB | 11.95 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/NLP-Translator-JA-EN/NLP-Translator-JA-EN
Repo clonata!
total 52
drwxr-xr-x 2 root root 4096 May  9 20:42 .
drwxr-xr-x 7 root root 4096 May  9 20:42 ..
-rw-r--r-- 1 root root 4470 May  9 20:42 dataset.py
-rw-r--r-- 1 root root 5061 May  9 20:42 evaluate.py
-rw-r--r-- 1 root root 2957 May  9 20:42 gui.py
-rw-r--r-- 1 root root    0 May  9 20:42 __init__.py
-rw-r--r-- 1 root root 2370 May  9 20:42 tokenizer.py
-rw-r--r-- 1 root root 7188 May  9 20:42 train.py
-rw-r--r-- 1 root root 9731 May  9 20:42 transformer.py


---
## 3. Parameters & path

In [51]:
PROJECT_PATH = "/content/drive/MyDrive/University/Project-NLP_Translator"
DATA_PATH = f"{PROJECT_PATH}/data/processed"
MODELS_PATH = f"{PROJECT_PATH}/models"

os.makedirs(MODELS_PATH, exist_ok=True)

EN_FILE = f"{DATA_PATH}/english.txt"
JP_FILE = f"{DATA_PATH}/japanese.txt"

#print(f"EN: {os.path.exists(EN_FILE)} ({os.path.getsize(EN_FILE)/1e9:.2f} GB)")
#print(f"JP: {os.path.exists(JP_FILE)} ({os.path.getsize(JP_FILE)/1e9:.2f} GB)")

# Parameters
VOCAB_SIZE = 32000
BATCH_SIZE = 16  # 32 OUT OF MEMORY
D_MODEL = 512
N_HEADS = 8
N_LAYERS = 6
D_FF = 2048
MAX_SAMPLES = 500000  # None = all dataset, or 1000000 for initial test

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

---
## 4. Tokenizer check on Drive

In [52]:
#DO NOT EXECUTE

import os

print("🔍 Ricerca english.txt e japanese.txt in tutto il Drive...")
print("=" * 60)

found_en = []
found_jp = []

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        full_path = os.path.join(root, file)
        if file == "english.txt":
            found_en.append(full_path)
            size = os.path.getsize(full_path) / 1e9
            print(f"✅ TROVATO english.txt:")
            print(f"   Path: {full_path}")
            print(f"   Size: {size:.2f} GB")
            print()
        elif file == "japanese.txt":
            found_jp.append(full_path)
            size = os.path.getsize(full_path) / 1e9
            print(f"✅ TROVATO japanese.txt:")
            print(f"   Path: {full_path}")
            print(f"   Size: {size:.2f} GB")
            print()

print("=" * 60)
if not found_en:
    print("❌ english.txt NON TROVATO da nessuna parte su Drive")
if not found_jp:
    print("❌ japanese.txt NON TROVATO da nessuna parte su Drive")

# Cerca anche con nomi simili
print("\n🔍 File .txt grandi (>1GB) nel Drive:")
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.txt'):
            full = os.path.join(root, file)
            try:
                size = os.path.getsize(full)
                if size > 1e9:
                    print(f"   {size/1e9:.2f} GB  ->  {full}")
            except:
                pass

🔍 Ricerca english.txt e japanese.txt in tutto il Drive...
✅ TROVATO japanese.txt:
   Path: /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/japanese.txt
   Size: 4.60 GB

✅ TROVATO english.txt:
   Path: /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/english.txt
   Size: 3.71 GB


🔍 File .txt grandi (>1GB) nel Drive:
   4.60 GB  ->  /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/japanese.txt
   3.71 GB  ->  /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/english.txt


In [53]:
print("=" * 50)
print("PATH VERIFY")
print("=" * 50)

# Check if file exists
en_exists = os.path.exists(EN_FILE)
jp_exists = os.path.exists(JP_FILE)

print(f"EN file: {EN_FILE}")
print(f"  Exists: {en_exists}")
if en_exists:
    print(f"  Dimension: {os.path.getsize(EN_FILE)/1e9:.2f} GB")

print(f"JP file: {JP_FILE}")
print(f"  Exists: {jp_exists}")
if jp_exists:
    print(f"  Dimension: {os.path.getsize(JP_FILE)/1e9:.2f} GB")

print(f"\nModels path: {MODELS_PATH}")
print(f"Device: {DEVICE}")

# Blocks everything if files do not exist
if not en_exists or not jp_exists:
    raise FileNotFoundError(
        "❌ Files not found on Drive!\n"
        "Verify that english.txt and japanese.txt are in:\n"
        f"{DATA_PATH}"
    )

print("\n✅ Every path is correct!")

PATH VERIFY
EN file: /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/english.txt
  Exists: True
  Dimension: 3.71 GB
JP file: /content/drive/MyDrive/University/Project-NLP_Translator/data/processed/japanese.txt
  Exists: True
  Dimension: 4.60 GB

Models path: /content/drive/MyDrive/University/Project-NLP_Translator/models
Device: cuda

✅ Every path is correct!


---
## 5. Import custom models & load existen tokenizer

In [54]:
from src.dataset import create_dataloaders
from src.transformer import Transformer, count_parameters
from src.train import Trainer, get_scheduler

print("✅ Moduls import success!")

✅ Moduls import success!


In [55]:
import sentencepiece as spm

# Load existing tokenizer in local
sp_en = spm.SentencePieceProcessor(model_file=f"{MODELS_PATH}/en_tokenizer.model")
sp_jp = spm.SentencePieceProcessor(model_file=f"{MODELS_PATH}/jp_tokenizer.model")

print(f"English tokenizer vocab: {sp_en.get_piece_size():,}")
print(f"Japanese tokenizer vocab: {sp_jp.get_piece_size():,}")

# Test rapido
test_en = "Hello, how are you?"
test_jp = "今日は良い天気ですね。"

en_ids = sp_en.encode(test_en, out_type=int, add_bos=True, add_eos=True)
jp_ids = sp_jp.encode(test_jp, out_type=int, add_bos=True, add_eos=True)

print(f"\nTest EN: '{test_en}'")
print(f"  Token IDs: {en_ids}")
print(f"  Decoded:   {sp_en.decode(en_ids)}")

print(f"\nTest JP: '{test_jp}'")
print(f"  Token IDs: {jp_ids}")
print(f"  Decoded:   {sp_jp.decode(jp_ids)}")

English tokenizer vocab: 32,000
Japanese tokenizer vocab: 32,000

Test EN: 'Hello, how are you?'
  Token IDs: [2, 9406, 22320, 298, 104, 61, 22344, 3]
  Decoded:   hello, how are you?

Test JP: '今日は良い天気ですね。'
  Token IDs: [2, 11812, 524, 7123, 2605, 6, 3]
  Decoded:   今日は良い天気ですね。


---
## 6. Dataset e DataLoader (lazy)

In [ ]:
# Estrai test set PRIMA di creare il dataloader (escludendo quegli indici)
# Oppure semplicemente usa file separati per train/test

In [56]:
# ============================================================
# DATASET E DATALOADER - DEFINITI DIRETTAMENTE NEL NOTEBOOK
# (bypass completo di src/dataset.py)
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from functools import partial


class LazyTranslationDataset(Dataset):
    """
    Dataset che NON carica il file in RAM.
    Usa 'rb' per l'indice (offset byte precisi) e decodifica in __getitem__.
    """
    def __init__(self, path_en, path_jp, sp_en, sp_jp, max_samples=None):
        self.sp_en = sp_en
        self.sp_jp = sp_jp
        self.path_en = path_en
        self.path_jp = path_jp

        # Build offset index in BINARY mode for accurate byte offsets
        self.offsets = []
        with open(path_en, 'rb') as f:
            while True:
                pos = f.tell()
                line = f.readline()
                if not line:
                    break
                self.offsets.append(pos)
                if max_samples and len(self.offsets) >= max_samples:
                    break

        self.len = len(self.offsets)
        print(f"LazyDataset: indexed {self.len:,} lines (RAM usage: ~{self.len * 8 / 1024 / 1024:.1f} MB)")

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        # Open in BINARY, seek to exact byte, then decode with error handling
        with open(self.path_en, 'rb') as f_en, \
             open(self.path_jp, 'rb') as f_jp:

            f_en.seek(self.offsets[idx])
            f_jp.seek(self.offsets[idx])

            # Read raw bytes and decode safely
            en_text = f_en.readline().decode('utf-8', errors='replace').strip()
            jp_text = f_jp.readline().decode('utf-8', errors='replace').strip()

        # Tokenize
        en_ids = self.sp_en.encode(en_text, out_type=int, add_bos=True, add_eos=True)
        jp_ids = self.sp_jp.encode(jp_text, out_type=int, add_bos=True, add_eos=True)

        return torch.tensor(en_ids, dtype=torch.long), torch.tensor(jp_ids, dtype=torch.long)


def collate_fn(batch, pad_idx=0):
    """
    Padding dinamico per batch.
    """
    src_batch, tgt_batch = zip(*batch)

    src_max = max(s.size(0) for s in src_batch)
    tgt_max = max(t.size(0) for t in tgt_batch)

    src_padded = torch.full((len(src_batch), src_max), pad_idx, dtype=torch.long)
    tgt_padded = torch.full((len(tgt_batch), tgt_max), pad_idx, dtype=torch.long)

    for i, (src, tgt) in enumerate(zip(src_batch, tgt_batch)):
        src_padded[i, :src.size(0)] = src
        tgt_padded[i, :tgt.size(0)] = tgt

    return src_padded, tgt_padded


def create_dataloaders(path_en, path_jp, sp_en, sp_jp, batch_size=32,
                       num_workers=0, max_samples=None):
    """
    Crea DataLoader con lazy loading.
    """
    dataset = LazyTranslationDataset(path_en, path_jp, sp_en, sp_jp, max_samples)

    collate = partial(collate_fn, pad_idx=0)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate,
        pin_memory=True if torch.cuda.is_available() else False
    )

    return loader


print("✅ Dataset e DataLoader definiti direttamente nel notebook!")

✅ Dataset e DataLoader definiti direttamente nel notebook!


In [57]:
train_loader = create_dataloaders(
    EN_FILE,
    JP_FILE,
    sp_en,
    sp_jp,
    batch_size=BATCH_SIZE,
    num_workers=0,
    max_samples=MAX_SAMPLES  # For sanity check: at the start use 100000
)

print(f"Train batches: {len(train_loader)}")

# Test: take a batch
batch_src, batch_tgt = next(iter(train_loader))
print(f"\nSource bach shape (EN): {batch_src.shape}")
print(f"Targhet bach shape (JP):   {batch_tgt.shape}")
print(f"Example src IDs: {batch_src[0][:15]}...")
print(f"Example tgt IDs: {batch_tgt[0][:15]}...")

LazyDataset: indexed 500,000 lines (RAM usage: ~3.8 MB)
Train batches: 31250

Source bach shape (EN): torch.Size([16, 63])
Targhet bach shape (JP):   torch.Size([16, 48])
Example src IDs: tensor([    2,   657, 12786,   281,  1053,    32,     8,  2932,    35,  2933,
        22333,   260,   195,   351,   988])...
Example tgt IDs: tensor([   2,    4,  425,  382, 4686,  862,  291,  135, 1498,  382, 2218,   33,
           6,    3,    0])...


---
## 7. Model: Transformer

In [58]:
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_encoder_layers=N_LAYERS,
    n_decoder_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=0.1,
    pad_idx=0
).to(DEVICE)

print(f"Model created!")
print(f"Total parameters: {count_parameters(model):,}")
print(f"Device: {DEVICE}")

Model created!
Total parameters: 93,322,496
Device: cuda


In [59]:
# Optimizer & scheduler
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9
)
scheduler = get_scheduler(optimizer, D_MODEL, warmup_steps=4000)

---
## 8. Training

In [60]:
# Fix per transformer.py: -1e9 overflowa in float16 (mixed precision)
# Sostituisce con -1e4 che è sufficiente per softmax

import re

transformer_path = '/content/NLP-Translator-JA-EN/src/transformer.py'

with open(transformer_path, 'r') as f:
    content = f.read()

# Sostituisci -1e9 con -1e4 nel masking
content = content.replace("scores.masked_fill(mask == 0, -1e9)",
                          "scores.masked_fill(mask == 0, -1e4)")

with open(transformer_path, 'w') as f:
    f.write(content)

print("✅ transformer.py patchato")
!grep -n "masked_fill" /content/NLP-Translator-JA-EN/src/transformer.py

✅ transformer.py patchato
58:            scores = scores.masked_fill(mask == 0, -1e4)


In [61]:
# Fix per train.py: aggiorna autocast e GradScaler

train_path = '/content/NLP-Translator-JA-EN/src/train.py'

with open(train_path, 'r') as f:
    content = f.read()

# Aggiorna import
content = content.replace(
    "from torch.cuda.amp import autocast, GradScaler",
    "from torch.amp import autocast, GradScaler"
)

# Aggiorna GradScaler
content = content.replace(
    "self.scaler = GradScaler()",
    "self.scaler = GradScaler('cuda')"
)

# Aggiorna autocast
content = content.replace(
    "with autocast():",
    "with autocast('cuda'):"
)

with open(train_path, 'w') as f:
    f.write(content)

print("✅ train.py patchato")
!grep -n "GradScaler\|autocast" /content/NLP-Translator-JA-EN/src/train.py

✅ train.py patchato
7:from torch.amp import autocast, GradScaler
44:        self.scaler = GradScaler('cuda')  # Mixed precision
70:            with autocast('cuda'):
127:            with autocast('cuda'):


In [62]:
import importlib
import sys

# Rimuovi cache
for mod in list(sys.modules.keys()):
    if 'src.' in mod:
        del sys.modules[mod]

# Re-importa
from src.transformer import Transformer, count_parameters
from src.train import Trainer, get_scheduler

print("✅ Moduli ricaricati con le fix")

✅ Moduli ricaricati con le fix


In [63]:
model = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_encoder_layers=N_LAYERS,
    n_decoder_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=0.1,
    pad_idx=0
).to(DEVICE)


print(f"Parametri: {count_parameters(model):,}")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
scheduler = get_scheduler(optimizer, D_MODEL, warmup_steps=4000)

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    save_dir=MODELS_PATH,
    log_interval=100
)

N_EPOCHS = 10

# Loads the last checkpoint saved
trainer.load_checkpoint(f"{MODELS_PATH}/checkpoint_epoch_0.pt")



# Forza a saltare l'epoca 0 già fatta
trainer.epoch = 1  # Inizia da epoca 1

print(f"Riprendo da epoca: {trainer.epoch}")

print(f"\n{'='*50}")
print(f"START TRAINING")
print(f"Epochs: {N_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Checkpoint saved in: {MODELS_PATH}")
print(f"{'='*50}\n")

# For sanity check: 2-3 epoch on 10% data
# For completed training: 10-20 epochs

# To continue from a prev checkpoint:
# trainer.load_checkpoint(f"{MODELS_PATH}/checkpoint_epoch_3.pt")

trainer.fit(n_epochs=N_EPOCHS)

Parametri: 93,322,496
Loaded checkpoint from epoch 0
Riprendo da epoca: 1

START TRAINING
Epochs: 10
Batch size: 16
Checkpoint saved in: /content/drive/MyDrive/University/Project-NLP_Translator/models



Epoch 1:   0%|          | 5/31250 [00:01<1:56:16,  4.48it/s, loss=9.2553, lr=1.75e-11]


OutOfMemoryError: CUDA out of memory. Tried to allocate 194.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 45.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.14 GiB is allocated by PyTorch, and 249.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## 9. Translation test

In [ ]:
def translate(text, direction="en-jp"):
    model.eval()
    with torch.no_grad():
        sp_src = sp_en if direction == "en-jp" else sp_jp
        sp_tgt = sp_jp if direction == "en-jp" else sp_en

        src_ids = sp_src.encode(text, out_type=int, add_bos=True, add_eos=True)
        src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)

        out = model.translate(src_tensor, max_len=50, bos_id=2, eos_id=3)
        out_ids = [id for id in out[0].cpu().tolist() if id not in [0, 2, 3]]
        return sp_tgt.decode(out_ids)

# Test
test_sentences = [
    "Hello, how are you today?",
    "I love programming using Java.",
    "The weather was nice the other day.",
    "Thank you very much for your present.",
    "Where is the train station?"
]

print("=" * 50)
for sent in test_sentences:
    translated = translate(sent, "en-jp")
    print(f"EN: {sent}")
    print(f"JP: {translated}")
    print("-" * 50)

---
## 10. BLEU evaluation & pre-trained model

In [ ]:
from src.evaluate import extract_test_set, evaluate_models

# Extract 1000 phrases for test model
test_en, test_jp, test_indices = extract_test_set(EN_FILE, JP_FILE, n=1000)

# Load best model
trainer.load_checkpoint(f"{MODELS_PATH}/best_model.pt")

# Evaluation EN -> JP
results_en_jp = evaluate_models(
    trainer.model,
    sp_en, sp_jp,
    test_en, test_jp,
    direction="en-jp"
    device=DEVICE
)

# Evaluation JP -> EN
results_jp_en = evaluate_models(
    trainer.model,
    sp_en, sp_jp,
    test_en, test_jp,
    direction="jp-en"
    device=DEVICE
)

# Save results
import json
with open(f"{MODELS_PATH}/evaluation_results.json", 'w') as f:
    json.dump({
        'en_jp': {
            'my_bleu': results_en_jp['my_bleu'],
            'pretrained_bleu': results_en_jp['pretrained_bleu']
        },
        'jp_en': {
            'my_bleu': results_jp_en['my_bleu'],
            'pretrained_bleu': results_jp_en['pretrained_bleu']
        }
    }, f, indent=2)

---
## 11. GUI on Colab

In [ ]:
# from src.gui import TranslatorApp

# app = TranslatorApp(trainer.model, sp_en, sp_jp, device='cuda')
# app.launch(share=True)  # Create temporary public link